# Lesson 4.3 — Deployment Strategies: Blue/Green, Rolling & Canary

**Level: Intermediate** | **Exam:** SOA-C03 Domain 3, Skills 3.1.3, 3.1.5; SAP-C02 Domain 2, Task 2.1

## Learning Objectives
- Implement and compare deployment strategies: in-place, rolling, blue/green, canary
- Configure AWS CodeDeploy for EC2 and ECS blue/green deployments
- Set up rollback triggers based on CloudWatch alarms
- Choose the right strategy for a given risk tolerance and availability requirement

_Reference: [AWS CodeDeploy](https://docs.aws.amazon.com/codedeploy/latest/userguide/) | [ECS Deployment Types](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/deployment-types.html)_

## Deployment Strategy Comparison

| Strategy | Downtime | Risk | Cost | Rollback Speed |
|----------|----------|------|------|---------------|
| **In-place (All-at-once)** | Yes | High | $ | Slow (redeploy) |
| **Rolling** | No | Medium | $ | Medium (partial) |
| **Rolling with additional batch** | No | Lower | $$ | Medium |
| **Immutable** | No | Low | $$$ | Fast (swap ASG) |
| **Blue/Green** | No | Very Low | $$$ | Instant (switch LB) |
| **Canary** | No | Lowest | $$$ | Instant (reroute traffic) |

## How Each Strategy Works

### Rolling
```
Original: [v1][v1][v1][v1]     
Step 1:   [v2][v1][v1][v1]     # Deploy to 1 instance
Step 2:   [v2][v2][v1][v1]     # Deploy to next
Step 3:   [v2][v2][v2][v1]
Final:    [v2][v2][v2][v2]     # All updated
```

### Blue/Green (AWS CodeDeploy or ECS)
```
Blue:  [v1][v1][v1][v1]  ← Production traffic (100%)
Green: [v2][v2][v2][v2]  ← New version (testing)
→ Switch: Route traffic to Green (instant)
→ Keep Blue for 1 hour → Rollback if needed → Terminate Blue
```

### Canary
```
All traffic: v1 (100%)
Deploy v2 canary:
  - 10% traffic → v2
  - 90% traffic → v1
  Wait + monitor alarms...
  - If OK: 100% → v2
  - If alarm: 100% → v1 (instant rollback)
```

## AWS Deployment Services

| Service | What it deploys | Strategy options |
|---------|----------------|------------------|
| **CodeDeploy** | EC2, ECS, Lambda, On-prem | In-place, Blue/Green |
| **Elastic Beanstalk** | Application code | All-at-once, Rolling, Immutable, Blue/Green |
| **ECS** | Container tasks | Rolling update, Blue/Green (with CodeDeploy) |
| **Lambda** | Function versions | Canary, Linear, All-at-once |
| **CodePipeline** | Orchestrates above | N/A (pipeline orchestrator) |

In [ ]:
import boto3, json

codedeploy = boto3.client('codedeploy', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
%%writefile appspec.yaml
# AppSpec file: tells CodeDeploy how to deploy your application
# This goes in the root of your application bundle (zip file)
version: 0.0
os: linux
files:
  - source: /app/
    destination: /var/www/myapp/
  - source: /config/nginx.conf
    destination: /etc/nginx/conf.d/

hooks:
  BeforeInstall:
    - location: scripts/stop_server.sh
      timeout: 30
      runas: root
  AfterInstall:
    - location: scripts/install_dependencies.sh
      timeout: 120
      runas: root
  ApplicationStart:
    - location: scripts/start_server.sh
      timeout: 60
      runas: root
  ValidateService:
    - location: scripts/health_check.sh
      timeout: 30
      runas: root

In [ ]:
# Explore CodeDeploy deployment configs
# Deployment configs define the % or count of instances to deploy to at once

configs = codedeploy.list_deployment_configs()['deploymentConfigsList']
print('Available CodeDeploy Deployment Configs:')
for config in sorted(configs):
    print(f'  {config}')

In [ ]:
# Lambda Canary Deployment via CodeDeploy
# Lambda supports traffic shifting between versions

print('Lambda Canary Deployment Configs:')
print()

canary_configs = [
    {'name': 'LambdaCanary10Percent5Minutes', 
     'desc': 'Shift 10% traffic to new version. After 5 min, shift 100%'},
    {'name': 'LambdaCanary10Percent10Minutes',
     'desc': 'Shift 10% for 10 minutes before full switchover'},
    {'name': 'LambdaLinear10PercentEvery1Minute',
     'desc': 'Shift 10% more every minute until 100%'},
    {'name': 'LambdaLinear10PercentEvery3Minutes',
     'desc': 'Shift 10% every 3 min — slower, more monitoring time'},
    {'name': 'LambdaAllAtOnce',
     'desc': 'Shift 100% immediately — fastest, highest risk'},
]

for c in canary_configs:
    print(f'  {c["name"]}')
    print(f'    → {c["desc"]}')

print()
print('With CloudWatch alarm rollback:')
print('If the alarm fires during the shift → CodeDeploy automatically rolls back to v1')

In [ ]:
# ECS Blue/Green Deployment with CodeDeploy
# Route traffic from old task set (Blue) to new task set (Green)

print('ECS Blue/Green Deployment Process:')
print('='*50)

process = [
    ('1', 'CodeDeploy creates new task set (Green) with v2 container'),
    ('2', 'Green is tested (can use test listener on port 8080)'),
    ('3', 'Canary/Linear traffic shift begins (e.g., 10% to Green)'),
    ('4', 'CloudWatch monitors error rate/latency on Green'),
    ('5a', 'If OK: 100% traffic to Green → Blue tasks stop (after bake time)'),
    ('5b', 'If ALARM: 100% traffic back to Blue (rollback)'),
]

for step, desc in process:
    print(f'  Step {step}: {desc}')

print()
print('Key CodeDeploy ECS config:')
eg_config = {
    'deploymentStyle': {
        'deploymentType': 'BLUE_GREEN',
        'deploymentOption': 'WITH_TRAFFIC_CONTROL'
    },
    'blueGreenDeploymentConfiguration': {
        'terminateBlueInstancesOnDeploymentSuccess': {
            'action': 'TERMINATE',
            'terminationWaitTimeInMinutes': 60    # Keep blue for 1hr (manual rollback window)
        },
        'deploymentReadyOption': {
            'actionOnTimeout': 'CONTINUE_DEPLOYMENT'
        }
    }
}
print(json.dumps(eg_config, indent=2))

## Exercise

In [ ]:
# TODO: For each scenario, choose the deployment strategy and justify it

scenarios = [
    {
        'app': 'Payment processing service — zero downtime required, instant rollback critical',
        'strategy': 'TODO',
        'reason': 'TODO'
    },
    {
        'app': 'Dev environment — fastest deploy, some downtime acceptable',
        'strategy': 'TODO',
        'reason': 'TODO'
    },
    {
        'app': 'New ML model scoring API — want to gradually expose to users, monitor accuracy',
        'strategy': 'TODO',
        'reason': 'TODO'
    },
    {
        'app': 'Lambda function update — small changes, monitored rollback if error rate increases',
        'strategy': 'TODO',
        'reason': 'TODO'
    },
]

for s in scenarios:
    print(f"App: {s['app']}")
    print(f"  Strategy: {s['strategy']}")
    print(f"  Reason: {s['reason']}")
    print()

In [ ]:
import os
if os.path.exists('appspec.yaml'):
    os.remove('appspec.yaml')
    print('Cleaned up appspec.yaml')

## Summary
- **All-at-once**: Fastest, cheapest, causes downtime — dev/test only
- **Rolling**: No downtime, gradual, medium risk — standard for stateless apps
- **Blue/Green**: No downtime, instant rollback, highest cost — best for production
- **Canary**: No downtime, minimal blast radius, gradual confidence building — best for high-traffic
- **CodeDeploy** supports automated rollback triggered by CloudWatch alarms — always configure this!

---
**Next:** [Lesson 4.4 — SSM OpsCenter, Parameter Store & Patch Manager](lesson-4.4-ssm-parameter-store.ipynb)